# Hyperparameter Tuning for Breast Cancer Classification

## Project Overview

The goal is to improve classification performance by selecting better model settings instead of relying only on default parameters.

The project uses the breast cancer diagnostic dataset. The target is recoded so that:

- `1 = malignant`
- `0 = benign`

This makes malignant tumors the positive class, which is useful because identifying malignant cases is usually the most important goal in a medical classification problem.

## Models Tuned

- Logistic Regression
- K Nearest Neighbors
- Support Vector Machine
- Decision Tree
- Random Forest
- Gradient Boosting

## Tuning Methods

- Baseline model comparison
- Grid Search Cross Validation
- Randomized Search Cross Validation
- Final test set evaluation

## Evaluation Metrics

- Accuracy
- Precision
- Recall
- F1 Score
- ROC AUC
- Confusion Matrix


## Why Hyperparameter Tuning?

Model parameters are learned from the data. Hyperparameters are settings chosen before training the model.

For example:

- KNN needs a value for `n_neighbors`
- Random Forest needs values for `n_estimators`, `max_depth`, and `min_samples_leaf`
- SVM needs values for `C`, `gamma`, and `kernel`

Poor hyperparameter choices can lead to underfitting or overfitting. Hyperparameter tuning helps find model settings that generalize better to unseen data.

### Import libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    RandomizedSearchCV
)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

### Load the dataset

In [3]:
df = pd.read_csv("breast_cancer_classification.csv")

df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,diagnosis,diagnosis_label
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0,malignant
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0,malignant
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0,malignant
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0,malignant
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0,malignant


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              569 non-null    float64
 1   mean texture             569 non-null    float64
 2   mean perimeter           569 non-null    float64
 3   mean area                569 non-null    float64
 4   mean smoothness          569 non-null    float64
 5   mean compactness         569 non-null    float64
 6   mean concavity           569 non-null    float64
 7   mean concave points      569 non-null    float64
 8   mean symmetry            569 non-null    float64
 9   mean fractal dimension   569 non-null    float64
 10  radius error             569 non-null    float64
 11  texture error            569 non-null    float64
 12  perimeter error          569 non-null    float64
 13  area error               569 non-null    float64
 14  smoothness error         569 non-null

In [5]:
df.shape

(569, 32)

In [6]:
df["diagnosis_label"].value_counts()

diagnosis_label
benign       357
malignant    212
Name: count, dtype: int64

The original dataset uses: 

0 = malignant, 1 = benign

Let's recode it to:

1 = malignant, 0 = benign

In [8]:
X = df.drop(columns=["diagnosis", "diagnosis_label"])

y = np.where(df["diagnosis_label"] == "malignant", 1, 0)

y = pd.Series(y, name="malignant")

In [9]:
y.value_counts()

malignant
0    357
1    212
Name: count, dtype: int64

In [10]:
y.value_counts(normalize=True)

malignant
0    0.627417
1    0.372583
Name: proportion, dtype: float64

### Train-test split

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training data shape:", X_train.shape)
print("Test data shape:", X_test.shape)

Training data shape: (455, 30)
Test data shape: (114, 30)


### An evaluation function

In [12]:
def evaluate_model(model_name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = model.decision_function(X_test)
    
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC AUC": roc_auc_score(y_test, y_proba)
    }

### Baseline models before tuning

In [13]:
baseline_models = {
    "Logistic Regression": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000, random_state=42))
        ]
    ),
    
    "KNN": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier())
        ]
    ),
    
    "SVM": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", SVC(probability=True, random_state=42))
        ]
    ),
    
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

In [14]:
baseline_results = []

for model_name, model in baseline_models.items():
    model.fit(X_train, y_train)
    
    results = evaluate_model(
        model_name=model_name,
        model=model,
        X_test=X_test,
        y_test=y_test
    )
    
    baseline_results.append(results)

baseline_results_df = pd.DataFrame(baseline_results)

baseline_results_df.sort_values(by="ROC AUC", ascending=False)

,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,Logistic Regression,0.964912,0.975000,0.928571,0.951220,0.996032
5,Gradient Boosting,0.964912,1.000000,0.904762,0.950000,0.994709
2,SVM,0.973684,1.000000,0.928571,0.962963,0.994709
4,Random Forest,0.973684,1.000000,0.928571,0.962963,0.992890
1,KNN,0.956140,0.974359,0.904762,0.938272,0.982308
3,Decision Tree,0.929825,0.904762,0.904762,0.904762,0.924603
